In [1]:
from syto.data.atlases.celfieish_atlases import CpGBetaCountsMethylationAtlas
import pandas as pd
import os
import json
import numpy as np
from syto.data.atlases.uxm_atlases import UXMMethylationAtlas
from syto.data.dataset_build.filters import pattern_length
from syto.classification.fit_data import load_columnar_split

In [2]:
reference_genome = "hg38"

In [73]:
old_data = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsTrainingData_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_no_data_leak_d041/"

In [74]:
for f in [old_data+x+".parquet" for x in ["train", "valid", "test"]]:
    print(f, pd.read_parquet(f).shape)

/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsTrainingData_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_no_data_leak_d041/train.parquet (2910114, 29)
/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsTrainingData_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_no_data_leak_d041/valid.parquet (915902, 29)
/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsTrainingData_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_no_data_leak_d041/test.parquet (900610, 29)


In [6]:
# data_path = f"/mnt/data/loyfer/parsed_dataset/U250.l4.{reference_genome}/staged"
data_path = f"/mnt/data/loyfer/parsed_dataset/U25.l4.{reference_genome}/labelers_on_train_no_pattern_len_filter/final"
original_atlas = UXMMethylationAtlas(atlas_name=f"UXMU25_{reference_genome}_l4", reference_genome=reference_genome, atlas_path=f"/home/luna.kuleuven.be/u0169940/Repos/UXM_deconv/supplemental/Atlas.U25.l4.{reference_genome}.full.tsv", ignore=["Megakaryocytes"])

In [4]:
declared_columns = ['chromosome',
 'read_start',
 'input_ids',
 'methylation_ids',
 'read_end',
 'original_label',
 'name',
 'region_start',
 'region_end',
 'dmr_ctype',
 'dmr_ctype_matched',
 'dmr_ctype_label',
 'file',
 'split']

In [9]:
original_atlas._atlas["NCPGS"] = original_atlas._atlas["endCpG"] - original_atlas._atlas["startCpG"] -1 

In [16]:
original_atlas._atlas.groupby("target")[["target", "NCPGS"]].agg(sum)["NCPGS"].min()

/tmp/ipykernel_4043367/3206827168.py:1: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  original_atlas._atlas.groupby("target")[["target", "NCPGS"]].agg(sum)["NCPGS"].min()


np.int64(49)

In [83]:
# data = pd.read_parquet(data_path, filters = [("name", "in", original_atlas._atlas["name"])])
data = load_columnar_split(dataset_dir=data_path, split="test", declared_columns=declared_columns)

In [5]:
labels_dict_path = "../../../App/labels_dict.json"
with open(labels_dict_path, "r", encoding="utf-8") as f:
            # JSON keys are strings; convert to {int: str}
            labels_dict = json.load(f)

In [19]:
data.reset_index(inplace=True)

In [7]:
data.rename(columns = {"chr":"chromosome", "trimmed_start":"read_start", "trimmed_end":"read_end"}, inplace=True)

In [85]:
data.rename(columns = {"methylation_ids":"pattern"}, inplace=True)

In [86]:
data["NCPGS"] = data["pattern"].map(pattern_length)

In [87]:
data[data["NCPGS"]>3].shape

(1339409, 15)

In [15]:
atlas = CpGBetaCountsMethylationAtlas.from_reads(data[data["NCPGS"]>3], reference_genome=reference_genome, atlas_name=f"UXMU25_{reference_genome}_l4", labels_dict=labels_dict, output_path=f"/mnt/data/loyfer/atlases/U25.l4.{reference_genome}.BetaCountsMethylAtlas.csv")

Aggregating groups: 100%|██████████| 35894/35894 [00:31<00:00, 1152.86group/s]


### Train only UXM atlas

In [26]:
train_only_uxm_atlas = UXMMethylationAtlas.from_reads(data[data["NCPGS"]>3], reference_genome=reference_genome, atlas_name=f"UXM_U25_{reference_genome}_l4_trainonly", 
                                       labels_dict=labels_dict, markers=original_atlas, 
                                       output_path=f"/mnt/data/loyfer/atlases/U25.l4.{reference_genome}_trainonly_uxm_atlas.csv")

In [30]:
data_path = f"/mnt/data/loyfer/parsed_dataset/U250.l4.{reference_genome}/staged"
data_all = pd.read_parquet(data_path, filters = [("name", "in", original_atlas._atlas["name"])])

In [32]:
data_all.rename(columns = {"methylation_ids":"pattern"}, inplace=True)
data_all["NCPGS"] = data_all["pattern"].map(pattern_length)

In [33]:
reconstructed_uxm_atlas = UXMMethylationAtlas.from_reads(data_all[data_all["NCPGS"]>3], reference_genome=reference_genome, atlas_name=f"UXM_U25_{reference_genome}_l4_trainonly", 
                                       labels_dict=labels_dict, markers=original_atlas)

In [54]:
def uxm_atlases_region_mae(atlas_a, atlas_b, exclude=None):
    """Per-region MAE between two UXM atlases over their cell-type columns.

    Regions are matched on ``name``; for each region the MAE is the mean
    absolute difference across the shared cell-type columns. Cell pairs where
    either atlas is NaN are skipped (so "no coverage" doesn't inflate the
    error); a region with no comparable cells gets NaN.

    Accepts UXMMethylationAtlas objects or plain DataFrames.

    Parameters
    ----------
    exclude : list of str, optional
        Cell-type columns to ignore (e.g. ["Megakaryocytes"]).

    Returns
    -------
    pd.DataFrame
        Indexed by region ``name``, with columns ``mae`` (per-region MAE) and
        ``n_cells`` (number of cell-type values compared).
    """
    meta = ["chr", "start", "end", "startCpG", "endCpG", "target", "direction"]
    exclude = set(exclude or [])

    def _df(x):
        return x.atlas if isinstance(x, UXMMethylationAtlas) else x

    a, b = _df(atlas_a).copy(), _df(atlas_b).copy()

    # align on shared regions
    shared = sorted(set(a["name"]) & set(b["name"]))
    a = a.set_index("name").reindex(shared)
    b = b.set_index("name").reindex(shared)

    # shared cell-type columns
    ct = [
        c for c in a.columns
        if c not in meta and c not in exclude and c in b.columns
    ]

    diff = np.abs(a[ct].to_numpy(dtype=float) - b[ct].to_numpy(dtype=float))
    with np.errstate(invalid="ignore"):  # rows that are all-NaN -> NaN mean
        mae = np.nanmean(diff, axis=1)
    n_cells = np.sum(~np.isnan(diff), axis=1)

    return pd.DataFrame({"mae": mae, "n_cells": n_cells}, index=pd.Index(shared, name="name"))

In [42]:
def uxm_atlases_approx_equal(atlas_a, atlas_b, atol=1e-3, exclude=None, verbose=True):
    """Sanity check that two UXM atlases are approximately equal.

    Regions are matched on ``name``. Metadata columns (chr, start, end,
    startCpG, endCpG, target, direction) must match exactly; cell-type
    fraction columns must agree within ``atol`` (NaN == NaN counts as equal).

    Accepts UXMMethylationAtlas objects or plain DataFrames. Returns bool.

    Parameters
    ----------
    exclude : list of str, optional
        Cell-type columns to ignore in the comparison (e.g. ["Megakaryocytes"],
        which the reference atlas has but a from_reads atlas may not).
    """
    meta = ["chr", "start", "end", "startCpG", "endCpG", "target", "direction"]
    exclude = set(exclude or [])

    def _df(x):
        return x.atlas if isinstance(x, UXMMethylationAtlas) else x

    a, b = _df(atlas_a).copy(), _df(atlas_b).copy()

    # same regions?
    if set(a["name"]) != set(b["name"]):
        if verbose:
            print(f"✗ region sets differ: "
                  f"{len(set(a['name']) - set(b['name']))} only in A, "
                  f"{len(set(b['name']) - set(a['name']))} only in B")
        return False

    # align on name
    a = a.set_index("name").sort_index()
    b = b.set_index("name").reindex(a.index)

    # metadata must match exactly (tolerant to int/float dtype)
    for c in meta:
        if c not in a or c not in b or not np.array_equal(
            a[c].to_numpy(), b[c].to_numpy()
        ):
            if verbose:
                print(f"✗ metadata column '{c}' differs")
            return False

    # cell-type columns must be the same set (excluding ignored ones)
    ct_a = [c for c in a.columns if c not in meta and c not in exclude]
    ct_b = [c for c in b.columns if c not in meta and c not in exclude]
    if set(ct_a) != set(ct_b):
        if verbose:
            print(f"✗ cell-type columns differ: "
                  f"only A={sorted(set(ct_a) - set(ct_b))}, "
                  f"only B={sorted(set(ct_b) - set(ct_a))}")
        return False

    # compare values within tolerance
    for c in ct_a:
        close = np.isclose(
            a[c].to_numpy(dtype=float), b[c].to_numpy(dtype=float),
            atol=atol, rtol=0.0, equal_nan=True,
        )
        if not close.all():
            if verbose:
                bad = a.index[~close][:5].tolist()
                print(f"✗ cell-type '{c}' differs at {int((~close).sum())} "
                      f"region(s), e.g. {bad}")
            return False

    if verbose:
        excl_note = f", excluding {sorted(exclude)}" if exclude else ""
        print(f"✓ atlases approximately equal "
              f"({len(a)} regions × {len(ct_a)} cell types, atol={atol}{excl_note})")
    return True